In this layer, the cleaned and standardized data from the Silver Layer is transformed into a dimensional model using a **Star Schema**, making it easier for business users and BI tools to analyze the data efficiently.

In [0]:
silver_path= "/Volumes/workspace/default/fmcg_data/silver/"
gold_path= "/Volumes/workspace/default/fmcg_data/gold/"

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [0]:
customers = spark.read.format("delta").load(silver_path + "customers")
products  = spark.read.format("delta").load(silver_path + "products")
stores    = spark.read.format("delta").load(silver_path + "stores")
orders    = spark.read.format("delta").load(silver_path + "orders")

In [0]:
# creating dim_customers
dim_customers = customers.select(
    "Customer_ID",
    "Customer_Name",
    "Gender",
    "Date_of_Birth",
    "Email",
    "Phone",
    "City",
    "State",
    "Country",
    "Loyalty_Status",
    "Join_Date"
).dropDuplicates(["Customer_ID"]) # duplicate drop needed to prevent dim_customers to not have duplicates in future

In [0]:
display(dim_customers.filter(col("Customer_ID").isNull()))

Customer_ID,Customer_Name,Gender,Date_of_Birth,Email,Phone,City,State,Country,Loyalty_Status,Join_Date


In [0]:
# display(dim_customers)

print("Rows:", dim_customers.count())

Rows: 850


In [0]:
dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path + "dim_customers")

In [0]:
# verifying
# display(dim_customers)
print(dim_customers.count())

850


In [0]:
# creating dim_products
dim_products = products.select(
    "Product_ID",
    "Product_Name",
    "Category",
    "Brand",
    "Supplier",
    "Unit_Price",
    "Launch_Date"
).dropDuplicates(["Product_ID"])

In [0]:
#display(dim_products)

print("Rows:", dim_products.count())

Rows: 270


In [0]:
dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path + "dim_products")

In [0]:
#verifying
#display(dim_products)
print(dim_products.count())

270


In [0]:
# creating dim_stores

dim_stores = stores.select(
    "Store_ID",
    "Store_Name",
    "City",
    "State",
    "Region",
    "Manager",
    "Opening_Date"
).dropDuplicates(["Store_ID"])

In [0]:
# display(dim_stores)

print("Rows:", dim_stores.count())

Rows: 43


In [0]:
dim_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path + "dim_stores")

creating fact_sales table now

In [0]:
# joining with products as it contains prices
fact_sales = (
    orders.alias("o")
    .join(
        products.select("Product_ID", "Unit_Price").alias("p"),
        on="Product_ID",
        how="left"
    )
)

In [0]:
# calculating sales amount
# Sales = Quantity × Unit Price × (1 − Discount/100)

fact_sales = fact_sales.withColumn("Sales_Amount",
    round(
        col("Quantity") * col("Unit_Price") * (1 - (col("Discount")/100)),2
    )
)

In [0]:
fact_sales = fact_sales.select(
    "Order_ID",
    "Order_Date",
    "Customer_ID",
    "Product_ID",
    "Store_ID",
    "Quantity",
    "Discount",
    "Payment_Mode",
    "Unit_Price",
    "Sales_Amount"
)

In [0]:
# verifying 
# display(fact_sales)

print("Rows:", fact_sales.count())

Rows: 17012


In [0]:
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path + "fact_sales")

In [0]:
# we need functions : col,dayofmonth, month, year, quarter, date_format 
# to create dim_time, we have imported all of them in beginning
# the date column is not standard it also shows time hence we need to just pick up date from the fact_sales 

dim_time = (
    fact_sales
    .select(to_date(col("Order_Date")).alias("Date"))
    .dropDuplicates()
    .withColumn("Day", dayofmonth(col("Date")))
    .withColumn("Day_Name", date_format(col("Date"), "EEEE"))
    .withColumn("Week", weekofyear(col("Date")))
    .withColumn("Month", month(col("Date")))
    .withColumn("Month_Name", date_format(col("Date"), "MMMM"))
    .withColumn("Quarter", quarter(col("Date")))
    .withColumn("Year", year(col("Date")))
)

In [0]:
# display(dim_time)

print("Rows:", dim_time.count())

Rows: 1196


In [0]:
dim_time.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path + "dim_time")

Using the following commands to export the gold tables as csv for Power BI Dashboards

In [0]:
export_path = "/Volumes/workspace/default/fmcg_data/powerbi/gold_after_increment"

tables = {
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_stores": dim_stores,
    "dim_time": dim_time,
    "fact_sales": fact_sales
}

for table_name, df in tables.items():
    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", "true")
          .csv(export_path + table_name)
    )

print("All Gold tables exported successfully!")

All Gold tables exported successfully!
